In [0]:
from classes.utils.utils import Utils

import sys
from datetime import datetime, timedelta, date
import pandas as pd

from pyspark.sql import functions as F, Window
from pyspark.sql.types import *

pd.set_option('display.float_format', lambda x: '%.5f' % x)

utils = Utils(dbutils, spark, prediction_date=True)
utils.get_prediction_date()
market_utils = utils.get_market_utils()

prediction_weeks = utils.prediction_weeks
training_weeks = utils.training_weeks
minimum_obs_required = utils.minimum_obs_required
max_sequential_missing_allowed = utils.max_sequential_missing_allowed

root_dir = utils.root_market_dir
#Get clean market name
market_name_clean = utils.market_name.split(' (', 1)[0]
market = market_name_clean

end_analysis_report_store = "/dbfs" + root_dir + "end_analysis_report/"

In [0]:
## Change these parameters according to the market
use_as_start_date = 'epos_sales_date'
use_as_end_date = 'epos_sales_date'

calculate_first_analysis = True
calculate_extra_analysis = True
calculate_drop_off = True
pre_work_feature_selection = True

In [0]:
print(utils.data_stores['raw'])

Translating the columnnames with help of the config mappings. This mapping needs to be changed per market probably. Use the initial data columns for the mapping in the config

After changing the config and/or market_utils, be sure to Clear the state and re-run the script to be sure all new changes are applied!

In [0]:
# Load in initial data for column mapping check
raw_data_path = market_utils.get_raw_data_path()
epos_data = market_utils.load_raw_data(raw_data_path)
if epos_data == None:
  print('No seperate files have been found. Change the get_raw_data_path() function for only one time to the correct initial data folder and change it back to original after this command has been re-runned.')
else:
  epos_data.display()

Loading in the initial epos_data and potential incremental data and do transformations

In [0]:
# Load in initial data and do transformations
raw_data_path = market_utils.get_raw_data_path()
epos_data = market_utils.load_raw_data(raw_data_path)
if epos_data is not None:
  epos_data = market_utils.market_raw_transformations(epos_data)
  print('Total sales_quantity after after initial read-in:', epos_data.agg(F.sum('epos_quantity')).collect()[0][0])

# Read the folder structure and select only the folders, not files
folders_list = market_utils.get_dir_content(raw_data_path)
folders_data = pd.DataFrame(folders_list, columns = ['folder'])
folders_data["copy_date"] = pd.Timestamp.now()

epos_data = market_utils.load_save_incremental_data(raw_data_path, folders_data, "load")
epos_data = epos_data.filter(F.col('eow_date') <= utils.prediction_date)

In [0]:
#Investigate the Raw Data if necessary
check_raw_data = True

cols = []

if check_raw_data:
  for folder in folders_data["folder"]:
    raw_file = spark.read.parquet(raw_data_path + folder)
    print(f'Showing data from: {folder}')
    print(f'Containing the following {len(raw_file.columns)} columns:\n{raw_file.columns}')
    if len(cols) > 0 and len(raw_file.columns) > len(cols):
      new_cols = [col for col in raw_file.columns if col not in cols]
      print(f'Of which these are new:\n {new_cols}')
    cols.extend(raw_file.columns)
    cols = list(set(cols))
    raw_file.display()

In [0]:
# Quick statistics check before doing the full analysis
epos_statistics = epos_data.groupBy('market', 'eow_date').agg(F.sum(F.col('epos_value')).alias('epos_value_per_date'), F.sum(F.col('epos_quantity')).alias('epos_quantity_per_date'), F.countDistinct(F.col('epos_product_id')).alias('n_products_sold'), F.countDistinct(F.col('epos_store_id')).alias('n_stores_sold')).orderBy('eow_date')

epos_statistics.display()

Databricks visualization. Run in Databricks to view.

EPOS data

Check if only correct market is in the dataset

In [0]:
epos_data.select('market').distinct().display()

In [0]:
epos_data.display()

Calculate date difference on the same row (if only sales date present, it will be always 0).

In [0]:
if calculate_extra_analysis:
  max_date_col = "epos_end_date" if "epos_end_date" in epos_data.columns else "epos_sales_date"
  date_diff = (
    epos_data.withColumn('epos_number_of_days', F.datediff(F.col(max_date_col), F.col('epos_sales_date')))
    .withColumn('day_of_week_start', F.dayofweek(F.col('epos_sales_date')))
    .withColumn('overlapping_week', F.when(F.weekofyear(F.col("epos_sales_date")) == F.weekofyear(F.col(max_date_col)), 'No').otherwise('Yes'))
     )
  date_diff = date_diff.groupBy(*[c for c in ['epos_number_of_days', 'store_chain', 'day_of_week_start', 'overlapping_week'] if c in date_diff.columns]).agg(F.count('epos_product_id').alias('count_rows')).orderBy('epos_number_of_days', 'day_of_week_start', 'overlapping_week')
  date_diff.display()

Calculate differences of sales dates between two consecutive rows

In [0]:
if calculate_extra_analysis:
  # Calculate difference in days until last sales
  window = Window.partitionBy('epos_product_id', 'epos_store_id').orderBy('epos_sales_date')

  sales_data = (
    epos_data.filter(F.col('epos_quantity') > 0) # Only positive sales are taken into account
        .withColumn("n_days_till_last_sales", (
          F.datediff(epos_data.epos_sales_date, F.lag(epos_data.epos_sales_date, 1).over(window))))
  )

In [0]:
if calculate_extra_analysis:
  # Count rounded average number of days till last sales per unique product/store combination (only 1 count per product/store combination)
  round_avg_count = (
    sales_data.groupBy('epos_store_id','epos_product_id').agg(F.round(F.mean('n_days_till_last_sales')).alias('round_avg_n_days_till_last_sales'))
    .groupBy('round_avg_n_days_till_last_sales').count()
    .orderBy('count', ascending = False)
  )

  round_avg_count.display()

In [0]:
if calculate_extra_analysis:
  # Count number of times that a unique product/store combination contains given number of days till last sales (multiple counts per product/store combination)
  total_count = (
    sales_data.dropDuplicates(['epos_store_id', 'epos_product_id', 'n_days_till_last_sales'])
    .groupBy('n_days_till_last_sales').count()
    .orderBy('count', ascending = False)
  )

  total_count.display()

In [0]:
if calculate_extra_analysis:
  # Most frequent number of days till last sales per unique product/store combination (only 1 count per product/store combination)
  w = Window.partitionBy('epos_store_id', 'epos_product_id').orderBy(F.col('count').desc())

  count_sales_data = (
    sales_data.groupBy('epos_store_id', 'epos_product_id', 'n_days_till_last_sales').count()
    .withColumn("row", F.row_number().over(w)).filter(F.col("row") == 1).drop("row")
  )

  frequent_count = count_sales_data.groupBy('n_days_till_last_sales').count().orderBy('count', ascending = False)

  frequent_count.display()

Results to get a better understanding of the data

In [0]:
#the sales dates
epos_data.select(F.col('epos_sales_date')).distinct().orderBy('epos_sales_date', ascending = False).display()

In [0]:
def export_to_csv(df):
  
  destination = root_dir + "numbers_of_stores_per_product_id"
  
  check = 0
  try:
    dbutils.fs.ls(destination)
  except:
    check = 1
  
  if check == 0:
    dbutils.fs.rm(destination, True)
  
  df.coalesce(1).write.format("csv").option("header", True).option("delimiter", ";").save(destination)

In [0]:
if calculate_extra_analysis:
  #stores per product id
  stores_per_product = epos_data.groupBy('epos_product_id').agg(F.countDistinct('epos_store_id').alias('n_stores')).orderBy('n_stores', ascending = False)
  stores_per_product.display()
  export_to_csv(stores_per_product)

In [0]:
if calculate_first_analysis:
  #function to get insight in the sales
  epos_statistics = epos_data.groupBy('eow_date').agg(F.sum(F.col('epos_value')).alias('epos_value_per_date'), F.sum(F.col('epos_quantity')).alias('epos_quantity_per_date'), F.countDistinct(F.col('epos_product_id')).alias('n_products_sold'), F.countDistinct(F.col('epos_store_id')).alias('n_stores_sold')).orderBy('eow_date')

  epos_statistics.display()

In [0]:
if calculate_first_analysis:
  epos_statistics.display()

Check differences in trend for multiple categories (Loop through several categories by adding them via Plot Options)

In [0]:
epos_data.display()

In [0]:
if calculate_first_analysis:
  if 'category' in epos_data.columns:
    epos_value_per_category = epos_data.groupBy('eow_date').pivot('category').sum('epos_value').orderBy('eow_date').na.fill(0)
    epos_value_per_category.display()

In [0]:
if calculate_first_analysis:
  if 'category' in epos_data.columns:
    epos_quantity_per_category = epos_data.groupBy('eow_date').pivot('category').sum('epos_quantity').orderBy('eow_date').na.fill(0)
    epos_quantity_per_category.display()

Monitor drop off of data during each step taken

In [0]:
def add_drop_off_row(data, stage, drop_off_table = None):
  drop_off_row = data.select(F.lit(stage).alias('Stage'), F.count("*").alias('n_rows'), F.sum('epos_quantity').alias('summed_quantity'), F.round(F.sum('epos_value'), 2).alias('summed_value'))
  if drop_off_table == None:
    return drop_off_row
  else:
    return total_drop_off_table.union(drop_off_row)

In [0]:
if calculate_drop_off:
  total_drop_off_table = add_drop_off_row(epos_data, 'Raw data')

  total_drop_off_table.display()

Functions used to calculate descriptive statistics

In [0]:
# Get the number of unique values for certain columns
def get_n_unique(df, market = None, customers=None):
  if not market:
    market = df.select('market').distinct().count()
  if not customers:
    customers = df.select('store_chain').distinct().count()
  
  N_unique_values = { 
    "The amount of unique countries in data": market,
     "The amount of unique customers in data": customers,
     "The amount of unique store id's in data": df.select('epos_store_id').distinct().count(),
     "The amount of unique product id's in data": df.select('epos_product_id').distinct().count(),
     "The amount of distinct rows":  df.distinct().count(),
     "The number of rows in the epos data": (df.count())
   }
  
  if 'category' in epos_data.columns:
    N_unique_values["The amount of unique categories in data"] = df.select('category').distinct().count()

  unique_values=pd.DataFrame(data=N_unique_values.values(), index=N_unique_values.keys())
  return unique_values


In [0]:
# Get the % of missing values for certain column
def get_null_value_stats(df):
  epos_data_nulls = df.select([F.when(F.col(c)=="",None).otherwise(F.col(c)).alias(c) for c in ['epos_product_id', 'epos_store_id', 'epos_value', 'epos_quantity']]).persist()
  total_rows = epos_data_nulls.count()
  nulls_bf = epos_data_nulls.select([(F.count(F.when(F.isnull(c), c))/total_rows).alias(c) for c in ['epos_product_id', 'epos_store_id', 'epos_value', 'epos_quantity']]).toPandas()

  N_null_values = {
    "Percentage of missing data in EAN": str(nulls_bf['epos_product_id'].values[0]*100)[:5]+'%',
    "Percentage of missing data in StoreCode": str(nulls_bf['epos_store_id'].values[0]*100)[:5]+'%',
    "Percentage of missing data in Sellout_Values": str(nulls_bf['epos_value'].values[0]*100)[:5]+'%',
    "Percentage of missing data in Sellout_Quantity": str(nulls_bf['epos_quantity'].values[0]*100)[:5]+'%'
  }

  null_values=pd.DataFrame(data=N_null_values.values(), index=N_null_values.keys())
  return null_values

In [0]:
# Get the timings of the data
def get_min_max_dates(df, date_col1 = use_as_start_date, date_col2 = use_as_end_date):
  min_date, max_date = df.select(F.min(date_col1), F.max(date_col2)).first()

  dates_info = {
    "Number of unique start dates": df.select(date_col1).distinct().count(),
    "Number of unique end dates": df.select(date_col2).distinct().count(),
    "First date present in the dataset": min_date,
    "Last date present in the dataset": max_date,
  }

  dates_info=pd.DataFrame(data=dates_info.values(), index=dates_info.keys())
  return dates_info

In [0]:
# Number of unique stores and products over several months
def get_monthly_sales_stats(df, date_col = use_as_end_date):
  monthly_sales = (df
      .groupBy(F.date_format(F.date_trunc('mm', F.col(date_col)),'yyyy-MM').alias("month"))
      .agg(F.countDistinct("epos_product_id").alias("approx_n_unique_products"),
           F.countDistinct("epos_store_id").alias("approx_n_unique_stores"))
      .orderBy('month')).toPandas()
  
  last_month = monthly_sales.iloc[-1]
  last_month = {
    "The amount of unique store id's last month({})".format(last_month['month']): last_month['approx_n_unique_stores'],
    "The amount of unique product id's last month({})".format(last_month['month']): last_month['approx_n_unique_products'],
  }

  last=pd.DataFrame(data=last_month.values(), index=last_month.keys())
  

  return last

In [0]:
# Get statistics about quantity and values sold
def get_min_max_stats(df):
  min_max = df.describe('epos_quantity', 'epos_value').toPandas()
  min_max.set_index("summary", inplace=True)
  minmax = {
    "Maximum value": min_max.loc["max"]["epos_value"],
    "Minimum value": min_max.loc["min"]["epos_value"],
    "Maximum quantity": min_max.loc["max"]["epos_quantity"],
    "Minimum quantity": min_max.loc["min"]["epos_quantity"],
    "Mean value": min_max.loc["mean"]["epos_value"],
    "Mean quantity": min_max.loc["mean"]["epos_quantity"],
    "Standard deviation of value": min_max.loc["stddev"]["epos_value"],
    "Standard deviation of quantity": min_max.loc["stddev"]["epos_quantity"],
  }

  minmax=pd.DataFrame(data=minmax.values(), index=minmax.keys())
  return minmax

In [0]:
# Get the number of duplicate values
def get_duplicates_df(df, date_col1 = use_as_start_date, date_col2 =use_as_end_date):
  if utils.market_abbreviation != 'ID':
    duplicates = df.groupBy("epos_store_id", "epos_product_id", date_col1, date_col2 )\
                  .count()\
                  .where(F.col('count') > 1)\
                  .select(F.sum('count')).toPandas()
  else:
    duplicates = df.groupBy("epos_store_id", "epos_product_id", "epos_bill_number", date_col1, date_col2 )\
                  .count()\
                  .where(F.col('count') > 1)\
                  .select(F.sum('count')).toPandas()
  
  duplicates.index=["Number of duplicated entries"]
  duplicates.rename(columns = {'sum(count)':0}, inplace = True)
  return duplicates

In [0]:
def get_total_value_2021_per_retailer(df, date_col = use_as_start_date):
  
  year_data = df.filter((F.col(date_col) >= '2021-01-01') & (F.col(date_col) <= '2021-12-31'))
  sales_per_retailer = year_data.groupby('store_chain').agg(F.sum("epos_value").alias('Total value')).toPandas()
#   sales_per_retailer = year_data.select(F.sum('epos_value')).toPandas()
  sales_per_retailer.set_index('store_chain', inplace = True)
  return sales_per_retailer

Get descriptive statistics for raw dataset

In [0]:
if calculate_first_analysis:
  N_unique_values_bf = get_n_unique(epos_data, 1)
  N_unique_values_bf

% of missing values per column

In [0]:
if calculate_first_analysis:
  null_values_bf = get_null_value_stats(epos_data)
  null_values_bf

Get the dates present in the dataset, to get an understanding from when to when this data is available

In [0]:
if calculate_first_analysis:
  dates_info_bf = get_min_max_dates(epos_data)
  dates_info_bf

Number of unique stores and products per month

In [0]:
if calculate_first_analysis:
  last_month_bf = get_monthly_sales_stats(epos_data)
  last_month_bf

In [0]:
if calculate_first_analysis:
  minmax_bf = get_min_max_stats(epos_data)
  minmax_bf

In [0]:
if calculate_first_analysis:
  duplicates_bf = get_duplicates_df(epos_data)
  duplicates_bf

In [0]:
if calculate_first_analysis:
  total_value_bf = get_total_value_2021_per_retailer(epos_data)
  total_value_bf

Decide on duplicate handeling

In [0]:
if calculate_first_analysis:
  #we want to know what datapoints are duplicates
  epos_data.groupBy("epos_sales_date", "epos_product_id", "epos_store_id").count().where(F.col('count') > 1).display()

In [0]:
utils.duplicate_handling_method

Decide on filters

In [0]:
if calculate_first_analysis:
  # First column is just for joining purposes, write down all filters we apply in the second row
  filters = {
    "First date present in the dataset": "outlier filter applied on quantity, only if quantity is > 500" ,
    "Last date present in the dataset": "drop rows where product_id or store_id is null or 0",
    "Maximum quantity": "drop duplicate rows based on product_id-store_id-start_date-end_date combination, sum the quantity and value",
    "Maximum value": "-",
    "Mean quantity": "-" ,
    "Mean value": "-",
    "Minimum quantity": "-",
    "Minimum value": "-",
    "Number of duplicated entries": "-",
    "Number of unique end dates": "-",
    "Number of unique start dates": "-",
    "Percentage of missing data in EAN": "-",
    "Percentage of missing data in Sellout_Quantity": "-",
    "Percentage of missing data in Sellout_Values": "-",
    "Percentage of missing data in StoreCode": "-",
    "Standard deviation of quantity": "-",
    "Standard deviation of value": "-",
    "The amount of distinct rows": "-",
    "The amount of unique countries in data": "-",
    "The amount of unique customers in data": "-",
    "The amount of unique product id's in data": "-",
    "{}".format(last_month_bf.index[1]): "-",
    "The amount of unique store id's in data": "-",
    "{}".format(last_month_bf.index[0]): "-",
    "The number of rows in the epos data": "-"
  }

  if 'category' in epos_data.columns:
    filters["The amount of unique categories in data"] = "-"

  if utils.duplicate_handling_method == 'max':
    filters['Maximum quantity'] = "drop duplicate rows based on product_id-start_date-end_date combination, take the maximum quantity and value"


  filters_df = pd.DataFrame(data = filters.values(), index = filters.keys())


Apply Filters

In [0]:
## Apply filters stated above on the dataset (except for the requirements part)

Delete product-store combinations with >= 7 days number_of_days. Change this for Ireland to be >= 8!

In [0]:
#product_store_ids_to_delete = epos_data.filter(F.col('epos_start_date') > '2021-01-01').filter(F.col('epos_number_of_days') >= 7).select('epos_product_id', #'epos_store_id').dropDuplicates().withColumn('delete_row', F.lit(1))

#epos_data_af = (epos_data
                  #.join(product_store_ids_to_delete, how='left', on=['epos_product_id', 'epos_store_id'])
                  #.fillna(0, subset = ["delete_row"])
                  #.filter(F.col('delete_row') != 1)
                  #.drop('delete_row')
               #)


In [0]:
epos_data_af = epos_data

In [0]:
#####################################################
## Replace null quantity and value with 0          ##
#####################################################
data = epos_data.fillna(0, subset=['epos_quantity', 'epos_value'])

#####################################################
## Remove duplicates, take sum/max value per group ##
#####################################################

w = Window.partitionBy(*utils.row_key_columns).orderBy(F.col("epos_quantity").desc())


if utils.duplicate_handling_method == 'max':
  data = data.withColumn("row",F.row_number().over(w)) \
                        .filter(F.col("row") == 1).drop("row")
  
if utils.duplicate_handling_method == 'sum':
  epos_data_max_rows = data.withColumn("row",F.row_number().over(w)) \
                        .filter(F.col("row") == 1).drop("row")

  summed_quantity_value = (
    data
      .groupBy(*utils.row_key_columns)
      .agg(
          F.sum('epos_quantity').alias('epos_quantity'),
          F.sum('epos_value').alias('epos_value')
      )
  )

  data = (epos_data_max_rows.drop('epos_quantity', 'epos_value')
                    .join(summed_quantity_value, how='left', on=utils.row_key_columns)
                 )
  
#####################################################
## Remove outliers                                 ##
#####################################################
  
hist_promo_weeks = 7
promotion_percentage = 10
min_trhrs_value = 500 # units
days = lambda i: i * 86400
window_ff_mean_dates = (
  Window()
    .partitionBy('epos_product_id', 'epos_store_id')
    .orderBy(F.col('eow_date').cast('timestamp').cast('long'))
    .rangeBetween(-days(7 * hist_promo_weeks), 0)
)


# Determine average quantity sold per store over the last X days
data = (
  data
    .withColumn("mean_quantity", F.mean(F.col('epos_quantity')).over(window_ff_mean_dates))
    .withColumn("std_quantity", F.stddev(F.col('epos_quantity')).over(window_ff_mean_dates))
    .withColumn("is_outlier", F.when((F.abs(F.col("epos_quantity")-F.col("mean_quantity")) > 3*F.col("std_quantity")) & (F.col('epos_quantity') > min_trhrs_value), 1).otherwise(0))
    .filter(F.col("is_outlier") == 0)
    .drop("mean_quantity", "std_quantity", "is_outlier")
).persist()

#####################################################
## Remove null and 0 for ids                       ##
#####################################################

if [c for c in data.schema if c.name == "epos_store_id"][0].dataType.typeName() == "string":
  data = data.filter((F.col('epos_product_id').isNotNull()) & (F.col('epos_product_id') != '0'))
  data = data.filter((F.col('epos_store_id').isNotNull()) & (F.col('epos_store_id') != '0'))

else:
  data = data.filter((F.col('epos_product_id').isNotNull()) & (F.col('epos_product_id') != 0))
  data = data.filter((F.col('epos_store_id').isNotNull()) & (F.col('epos_store_id') != 0))

epos_data_af = data.persist()

In [0]:
if calculate_drop_off:
  total_drop_off_table = add_drop_off_row(epos_data_af, 'Filtered data', total_drop_off_table)

  total_drop_off_table.display()

In [0]:
epos_data_af.display()

Descriptive statistics filtered data

In [0]:
if calculate_first_analysis:
  unique_values_af = get_n_unique(epos_data_af, 1)
  null_values_af = get_null_value_stats(epos_data_af)
  dates_info_af = get_min_max_dates(epos_data_af)
  last_month_af = get_monthly_sales_stats(epos_data_af)
  minmax_af = get_min_max_stats(epos_data_af)
  duplicates_af = get_duplicates_df(epos_data_af)
  total_value_af = get_total_value_2021_per_retailer(epos_data_af)

Effect of taking last 6 months

In [0]:
dates = [item.eow_date for item in epos_data_af.select('eow_date').distinct().orderBy('eow_date', ascending=False).collect()]
TRAINING_DATES = dates[prediction_weeks:prediction_weeks+training_weeks]
PREDICTION_DATES = dates[0:prediction_weeks]
epos_data_af_last_6_months = epos_data_af.filter(F.col('eow_date').isin(TRAINING_DATES + PREDICTION_DATES))

In [0]:
dates

In [0]:
if calculate_first_analysis:
  unique_values_af_last_6_months = get_n_unique(epos_data_af_last_6_months, 1)
  null_values_af_last_6_months = get_null_value_stats(epos_data_af_last_6_months)
  dates_info_af_last_6_months = get_min_max_dates(epos_data_af_last_6_months)
  last_month_af_last_6_months = get_monthly_sales_stats(epos_data_af_last_6_months)
  minmax_af_last_6_months = get_min_max_stats(epos_data_af_last_6_months)
  duplicates_af_last_6_months = get_duplicates_df(epos_data_af_last_6_months)
  #total_value_af_last_6_months = get_total_value_2021_per_retailer(epos_data_af_last_6_months)

In [0]:
if calculate_first_analysis:
  total_value_af_last_6_months = get_total_value_2021_per_retailer(epos_data_af_last_6_months)
  total_value_af_last_6_months

In [0]:
if calculate_drop_off:
  total_drop_off_table = add_drop_off_row(epos_data_af_last_6_months, 'Last 6 months', total_drop_off_table)

  total_drop_off_table.display()

Check the requirements of data for the model and check the number of unique products and stores

In [0]:
if calculate_first_analysis:
  filter_requirements = {
    "First date present in the dataset": "Aggregated to weekly data"  ,
    "Last date present in the dataset": "20 weeks of sales (>0 quantity)" ,
    "Maximum quantity":  "<= 4 weeks of no sales concurrently",
    "Maximum value": "-",
    "Mean quantity": "-",
    "Mean value": "-",
    "Minimum quantity": "-",
    "Minimum value": "-", 
    "Number of duplicated entries": "-" ,
    "Number of unique end dates": "-",
    "Number of unique start dates": "-",
    "Percentage of missing data in EAN": "-",
    "Percentage of missing data in Sellout_Quantity": "-",
    "Percentage of missing data in Sellout_Values": "-",
    "Percentage of missing data in StoreCode": "-",
    "Standard deviation of quantity": "-",
    "Standard deviation of value": "-",
    "The amount of distinct rows": "-",
    "The amount of unique countries in data": "-",
    "The amount of unique customers in data": "-",
    "The amount of unique product id's in data": "-",
    "{}".format(last_month_bf.index[1]): "-",
    "The amount of unique store id's in data": "-",
    "{}".format(last_month_bf.index[0]): "-",
    "The number of rows in the epos data": "-"
  }

  if 'category' in epos_data.columns:
    filter_requirements["The amount of unique categories in data"] = "-"

  filter_requirements_df = pd.DataFrame(data = filter_requirements.values(), index = filter_requirements.keys())

In [0]:
#####################################################
## Replace null quantity and value with 0          ##
#####################################################
epos_data_af_last_6_months = epos_data_af_last_6_months.fillna(0, subset=['epos_quantity', 'epos_value'])

# Aggregate to weekly level
epos_data_weekly = (
  epos_data_af_last_6_months
    .groupBy(*[c for c in ['market', 'epos_product_id', 'store_chain', 'epos_store_id', 'eow_date', 'category'] if c in data.columns])
    .agg(
        F.sum('epos_quantity').alias('epos_quantity'),
        F.sum('epos_value').alias('epos_value'),
    )
)

if calculate_drop_off:
  total_drop_off_table = add_drop_off_row(epos_data_weekly, 'Last 6 months weekly', total_drop_off_table)

  total_drop_off_table.display()


Apply requirements

In [0]:
TRAINING_DATES

In [0]:
first_sold_check = (epos_data_af
        .groupBy('epos_store_id', 'epos_product_id')
        .agg(F.min('eow_date').alias('first_eow_date_available'))
        .withColumn('sold_before_training_set', F.when(F.col('first_eow_date_available') <= TRAINING_DATES[-1], 1).otherwise(0))
       )

first_sold_check.display()

In [0]:
data_training_weeks = (epos_data_weekly
        .join(first_sold_check, on=['epos_store_id', 'epos_product_id'], how='left')
        .filter(F.col('sold_before_training_set') == 1)
       )

if calculate_drop_off:
  total_drop_off_table = add_drop_off_row(data_training_weeks, 'Requirement: on shelves for 26 weeks', total_drop_off_table)

  total_drop_off_table.display()

In [0]:
#####################################################
## 4. Filtering data based on requirements         ##
#####################################################
#Make conditional count function for counting the non-null quantities
cnt_cond = lambda cond: F.sum(F.when(cond, 1).otherwise(0))

correctNegativeToZero = F.udf(lambda col: 0 if col < 0 else col, IntegerType())

# filter on time period used for model development
data = data_training_weeks.filter(F.col('eow_date').isin(TRAINING_DATES + PREDICTION_DATES))

# Set negative weekly quantities to 0
data = data.withColumn('epos_quantity', correctNegativeToZero(data.epos_quantity))


# Calculate difference in weeks until last sales
window = Window.partitionBy('epos_product_id', 'epos_store_id').orderBy('eow_date')

sales_data = (
  data.filter(F.col('epos_quantity') > 0) # Only positive sales are taken into account
      .withColumn("n_weeks_till_last_sales", (
        F.datediff(data.eow_date, F.lag(data.eow_date, 1).over(window))/7))
)

# determine which product stores meet the data requirements
requirements = (
  sales_data
    .groupBy('epos_product_id', 'epos_store_id')
    # Count number of observations and maximum number of missing observations
      .agg(
        cnt_cond(F.col('epos_quantity').isNotNull()).alias('n_obs'),   
        (F.max('n_weeks_till_last_sales')).alias('sequential_missing_values')
      )
)

# join requirements data with original data
data_requirements = (
  data
    .join(requirements, on = ['epos_product_id', 'epos_store_id'], how='left')
)

In [0]:
# Filter based on the number of observations needed
data_requirements_obs = (data_requirements
        .filter(F.col('n_obs') >= minimum_obs_required)
       )

if calculate_drop_off:
  total_drop_off_table = add_drop_off_row(data_requirements_obs, 'Requirement: 20 weeks of sales', total_drop_off_table)

  total_drop_off_table.display()

In [0]:
data_requirements_seq_missing = (data_requirements_obs
        .filter(F.col('sequential_missing_values') <= max_sequential_missing_allowed)
       )

if calculate_drop_off:
  total_drop_off_table = add_drop_off_row(data_requirements_seq_missing, 'Requirement: max 4 weeks consecutive no sales', total_drop_off_table)

  total_drop_off_table.display()

In [0]:
n_stores_sold = (
                  data_requirements_seq_missing
                  .groupBy('epos_product_id')
                  .agg(F.countDistinct('epos_store_id').alias('n_stores_sold')))

requirements_with_n_stores = (
                data_requirements_seq_missing
                .join(n_stores_sold, on = ['epos_product_id'], how = 'left'))

# create subset and clean up; 
epos_data_with_requirements = (
    requirements_with_n_stores
    .filter(F.col('n_stores_sold') >= 20)
    .drop('max_date', 'n_obs', 'sequential_missing_values')
).persist()

if calculate_drop_off:
  total_drop_off_table = add_drop_off_row(epos_data_with_requirements, 'Requirement: min 20 stores sold after requirements', total_drop_off_table)

  total_drop_off_table.display()

Filter To get Total Value on Epos Data with Requirements

In [0]:
def get_total_value_data_with_requirements():

  #first for one half of the year 2021

  # quantity and price are the anomalized quantity and price. 
  # Hence, if certain product/store/date combination has been predicted correctly, the quantity and price will be interpolated. Otherwise: original quantity is taken
  # Aggregate to weekly level
  epos_data_weekly = (
    epos_data_af
      .groupBy(*[c for c in ['market', 'epos_product_id', 'store_chain', 'epos_store_id', 'eow_date', 'category'] if c in epos_data_af.columns])
      .agg(
          F.sum('epos_quantity').alias('epos_quantity'),
          F.sum('epos_value').alias('epos_value'),
      )
  )

  
  
  

  #####################################################
  ## 3. Split train & test                           ##
  #####################################################


  dates = [item.eow_date for item in epos_data_af.filter(F.col('eow_date').isNotNull()).select('eow_date').distinct().orderBy('eow_date', ascending=False).collect()]
  dates = [day for day in dates if day < date(2022,1,1)]
 

  TRAINING_DATES = dates[prediction_weeks:prediction_weeks+training_weeks]
  PREDICTION_DATES = dates[0:prediction_weeks]
  
  
  first_sold_check = (epos_data_af
        .groupBy('epos_store_id', 'epos_product_id')
        .agg(F.min('eow_date').alias('first_eow_date_available'))
        .withColumn('sold_before_training_set', F.when(F.col('first_eow_date_available') <= TRAINING_DATES[-1], 1).otherwise(0))
       )

  #first_sold_check.display()
  
  data_training_weeks = (epos_data_weekly
        .join(first_sold_check, on=['epos_store_id', 'epos_product_id'], how='left')
        .filter(F.col('sold_before_training_set') == 1)
       )
  
  

  from datetime import timedelta

  #####################################################
  ## 4. Filtering data based on requirements         ##
  #####################################################
  #Make conditional count function for counting the non-null quantities
  cnt_cond = lambda cond: F.sum(F.when(cond, 1).otherwise(0))

  correctNegativeToZero = F.udf(lambda col: 0 if col < 0 else col, IntegerType())

  # filter on time period used for model development
  data = data_training_weeks.filter(F.col('eow_date').isin(TRAINING_DATES + PREDICTION_DATES))

  # Set negative weekly quantities to 0
  data = data.withColumn('epos_quantity', correctNegativeToZero(data.epos_quantity))


  # Calculate difference in weeks until last sales
  window = Window.partitionBy('epos_product_id', 'epos_store_id').orderBy('eow_date')

  sales_data = (
    data.filter(F.col('epos_quantity') > 0) # Only positive sales are taken into account
        .withColumn("n_weeks_till_last_sales", (
          F.datediff(data.eow_date, F.lag(data.eow_date, 1).over(window))/7))
  )

  # determine which product stores meet the data requirements
  requirements = (
    sales_data
      .groupBy('epos_product_id', 'epos_store_id')
      # Count number of observations and maximum number of missing observations
        .agg(
          cnt_cond(F.col('epos_quantity').isNotNull()).alias('n_obs'),   
          (F.max('n_weeks_till_last_sales')).alias('sequential_missing_values')
        )
  )

  # join requirements data with original data
  data_requirements = (
    data
      .join(requirements, on = ['epos_product_id', 'epos_store_id'], how='left')
      .filter(F.col('n_obs') >= minimum_obs_required)
      .filter(F.col('sequential_missing_values') <= max_sequential_missing_allowed)
  )

  # Get the number of stores sold in after the requirements applied
  n_stores_sold = (
                    data_requirements
                    .groupBy('epos_product_id')
                    .agg(F.countDistinct('epos_store_id').alias('n_stores_sold')))

  requirements_with_n_stores = (
                  data_requirements
                  .join(n_stores_sold, on = ['epos_product_id'], how = 'left'))

  # Filter on minimum 20 stores sold
  epos_data_with_requirements = (
      requirements_with_n_stores
      .filter(F.col('n_stores_sold') >= 20)
      .drop('max_date', 'n_obs', 'sequential_missing_values')
  ).persist()
  
  temp_total_1 = get_total_value_2021_per_retailer(epos_data_with_requirements, "eow_date")
  
  ##############
  
  
  #now for the other half year

  #####################################################
  ## 3. Split train & test                           ##
  #####################################################


  dates = [item.eow_date for item in epos_data_af.filter(F.col('eow_date').isNotNull()).select('eow_date').distinct().orderBy('eow_date', ascending=False).collect()]
  dates = [day for day in dates if day < date(2021,7,1)]

  TRAINING_DATES = dates[prediction_weeks:prediction_weeks+training_weeks]
  PREDICTION_DATES = dates[0:prediction_weeks]
  
  
  first_sold_check = (epos_data_af
        .groupBy('epos_store_id', 'epos_product_id')
        .agg(F.min('eow_date').alias('first_eow_date_available'))
        .withColumn('sold_before_training_set', F.when(F.col('first_eow_date_available') <= TRAINING_DATES[-1], 1).otherwise(0))
       )

  #first_sold_check.display()
  
  data_training_weeks = (epos_data_weekly
        .join(first_sold_check, on=['epos_store_id', 'epos_product_id'], how='left')
        .filter(F.col('sold_before_training_set') == 1)
       )
  

  from datetime import timedelta

  #####################################################
  ## 4. Filtering data based on requirements         ##
  #####################################################
  #Make conditional count function for counting the non-null quantities
  cnt_cond = lambda cond: F.sum(F.when(cond, 1).otherwise(0))

  correctNegativeToZero = F.udf(lambda col: 0 if col < 0 else col, IntegerType())

  # filter on time period used for model development
  data = data_training_weeks.filter(F.col('eow_date').isin(TRAINING_DATES + PREDICTION_DATES))

  # Set negative weekly quantities to 0
  data = data.withColumn('epos_quantity', correctNegativeToZero(data.epos_quantity))


  # Calculate difference in weeks until last sales
  window = Window.partitionBy('epos_product_id', 'epos_store_id').orderBy('eow_date')

  sales_data = (
    data.filter(F.col('epos_quantity') > 0) # Only positive sales are taken into account
        .withColumn("n_weeks_till_last_sales", (
          F.datediff(data.eow_date, F.lag(data.eow_date, 1).over(window))/7))
  )

  # determine which product stores meet the data requirements
  requirements = (
    sales_data
      .groupBy('epos_product_id', 'epos_store_id')
      # Count number of observations and maximum number of missing observations
        .agg(
          cnt_cond(F.col('epos_quantity').isNotNull()).alias('n_obs'),   
          (F.max('n_weeks_till_last_sales')).alias('sequential_missing_values')
        )
  )

  # join requirements data with original data
  data_requirements = (
    data
      .join(requirements, on = ['epos_product_id', 'epos_store_id'], how='left')
      .filter(F.col('n_obs') >= minimum_obs_required)
      .filter(F.col('sequential_missing_values') <= max_sequential_missing_allowed)
  )

  # Get the number of stores sold in after the requirements applied
  n_stores_sold = (
                    data_requirements
                    .groupBy('epos_product_id')
                    .agg(F.countDistinct('epos_store_id').alias('n_stores_sold')))

  requirements_with_n_stores = (
                  data_requirements
                  .join(n_stores_sold, on = ['epos_product_id'], how = 'left'))

  # Filter on minimum 20 stores sold
  epos_data_with_requirements = (
      requirements_with_n_stores
      .filter(F.col('n_stores_sold') >= 20)
      .drop('max_date', 'n_obs', 'sequential_missing_values')
  ).persist()
  
  temp_total_2 = get_total_value_2021_per_retailer(epos_data_with_requirements, "eow_date") 

  year_total = pd.concat([temp_total_1, temp_total_2], axis=1).fillna(0).sum(axis=1)
  year_total = year_total.to_frame()
  year_total.columns = ['Total value']
  
  return year_total
  

In [0]:
dates

In [0]:
# Number of unique stores and products over several months
def get_monthly_sales_stats(df, date_col = use_as_end_date):
  monthly_sales = (df
      .groupBy(F.date_format(F.date_trunc('mm', F.col(date_col)),'yyyy-MM').alias("month"))
      .agg(F.countDistinct("epos_product_id").alias("approx_n_unique_products"),
           F.countDistinct("epos_store_id").alias("approx_n_unique_stores"))
      .orderBy('month')).toPandas()
  
  last_month = monthly_sales.iloc[-1]
  last_month = {
    "The amount of unique store id's last month({})".format(last_month['month']): last_month['approx_n_unique_stores'],
    "The amount of unique product id's last month({})".format(last_month['month']): last_month['approx_n_unique_products'],
  }

  last=pd.DataFrame(data=last_month.values(), index=last_month.keys())
  

  return last

In [0]:
if calculate_first_analysis:
  unique_values_af_wr = get_n_unique(epos_data_with_requirements,1)
  null_values_af_wr = get_null_value_stats(epos_data_with_requirements)
  dates_info_af_wr = get_min_max_dates(epos_data_with_requirements, "eow_date", "eow_date")
  last_month_af_wr = get_monthly_sales_stats(epos_data_with_requirements, "eow_date")
  minmax_af_wr = get_min_max_stats(epos_data_with_requirements)
  duplicates_af_wr = get_duplicates_df(epos_data_with_requirements, "eow_date", "eow_date")

if calculate_extra_analysis:
  total_value_af_wr = get_total_value_data_with_requirements()

In [0]:
duplicates = epos_data_with_requirements.groupBy("epos_store_id", "epos_product_id", "eow_date")\
                .count()

In [0]:
duplicates.display()

Merging results

In [0]:
if calculate_first_analysis:
  df_bf  = pd.concat([N_unique_values_bf, null_values_bf, last_month_bf, dates_info_bf, duplicates_bf, minmax_bf])
  df_bf

In [0]:
if calculate_first_analysis:
  df_af = pd.concat([unique_values_af, null_values_af, last_month_af, dates_info_af, duplicates_af, minmax_af])

In [0]:
if calculate_first_analysis:
  df_af_last_6_months = pd.concat([unique_values_af_last_6_months, null_values_af_last_6_months, last_month_af_last_6_months, dates_info_af_last_6_months, duplicates_af_last_6_months, minmax_af_last_6_months])

In [0]:
if calculate_first_analysis:
  df_af_wr = pd.concat([unique_values_af_wr, null_values_af_wr, last_month_af_wr, dates_info_af_wr, duplicates_af_wr, minmax_af_wr])

In [0]:
if calculate_first_analysis:
  df_merged = pd.merge(df_bf, filters_df, left_index = True, right_index = True, how = 'outer')
  df_merged = pd.merge(df_merged, df_af,left_index=True, right_index=True, how = 'outer')
  df_merged = pd.merge(df_merged, df_af_last_6_months,left_index=True, right_index=True, how = 'outer')
  df_merged = pd.merge(df_merged, filter_requirements_df,left_index=True, right_index=True, how = 'outer')
  df_merged = pd.merge(df_merged, df_af_wr,left_index=True, right_index=True, how = 'outer')
  df_merged.columns = ["Raw","Filters applied", "Statistics after filters applied", "Last 6 months after filters applied","requirements applied","Statistics after filters and requirements applied"]
  
  df_merged
  

In [0]:
if calculate_first_analysis:
  total_value_merged = pd.merge(total_value_bf, total_value_af, left_index = True, right_index = True, how = "outer")
  total_value_merged = pd.merge(total_value_merged, total_value_af_wr, left_index = True, right_index = True, how = "outer")
  total_value_merged.columns = ["Total value raw", "Total value after filter applied", "Total value after filter and requirements applied"]
  total_value_merged

In [0]:
if calculate_extra_analysis:
  epos_statistics_with_requirements = epos_data_with_requirements.groupBy('eow_date').agg(F.sum(F.col('epos_value')).alias('epos_value_per_date'), F.sum(F.col('epos_quantity')).alias('epos_quantity_per_date'), F.countDistinct(F.col('epos_product_id')).alias('n_products_sold'), F.countDistinct(F.col('epos_store_id')).alias('n_stores_sold')).orderBy('eow_date')

  epos_statistics_with_requirements.display()

In [0]:
if calculate_extra_analysis:
  epos_statistics_with_requirements.display()

Exporting excel file

In [0]:
if calculate_extra_analysis:
  #Export total values
  root_dir = utils.root_market_dir

  destination = root_dir + "Total_values_2021"
  #destination_file = "/dbfs" + destination_directory + '/' +  "total_values_2021.csv"
  #filename = "total_values_2021.csv"

  check = 0
  try:
    dbutils.fs.ls(destination)
  except:
    check = 1

  if check == 0:
    dbutils.fs.rm(destination, True)

  #filename =  "total_values_2021.csv"
  #path = "/dbfs" + destination + "/" + filename
  #total_value_merged.to_csv(path, sep = ';')
  #export_frame = spark.createDataFrame(total_value_merged)

  total_value_merged.index.name = 'store_chain'
  temp_frame = total_value_merged.reset_index(drop = False)
  export_frame = spark.createDataFrame(temp_frame)
  export_frame = export_frame.select('store_chain', F.format_string('%.2f', F.round(export_frame['Total value raw'], 2)).alias('Total value raw'), F.format_string('%.2f', F.round(export_frame['Total value after filter applied'], 2)).alias('Total value after filter applied'), F.format_string('%.2f', F.round(export_frame['Total value after filter and requirements applied'], 2)).alias('Total value after filter and requirements applied'))
  export_frame.display()
  export_frame.coalesce(1).write.format("csv").option("header", True).option("delimiter", ";").save(destination)

In [0]:
if calculate_extra_analysis:
  #Export descriptive statistics
  import os

  loc_save = "/tmp/end_analysis_report_" + market + ".xlsx"

  os.environ['SAVE_LOC_LOCAL'] = loc_save
  os.environ['SAVE_LOC_BLOB'] = end_analysis_report_store

  writer = pd.ExcelWriter(loc_save, engine='xlsxwriter')
  df_merged.to_excel(writer, sheet_name=market, startrow=1, index=True)
  writer.save()

In [0]:
%sh
ls

In [0]:
%sh
rm -r "$SAVE_LOC_BLOB"
mkdir "$SAVE_LOC_BLOB"
sudo mv "$SAVE_LOC_LOCAL" "$SAVE_LOC_BLOB"

### Check representative products to use for feature engineering

Apply promo split

In [0]:
dates = [item.eow_date for item in epos_data.select('eow_date').distinct().orderBy('eow_date', ascending=False).collect()]
TRAINING_DATES = dates[prediction_weeks:prediction_weeks+training_weeks]
PREDICTION_DATES = dates[0:prediction_weeks]

weekly_data = (
  epos_data_af
    .groupBy(*[c for c in ['epos_product_id', 'epos_store_id', 'category', 'eow_date', 'brand'] if c in data.columns])
    .agg(
        F.sum('epos_quantity').alias('epos_quantity'),
        F.sum('epos_value').alias('epos_value'),
        F.countDistinct('epos_sales_date').alias("n_sales_days")
    )
)

epos_data_with_requirements = market_utils.apply_requirements(weekly_data, TRAINING_DATES, PREDICTION_DATES)

In [0]:
# Calculate promotions for the promo split
window_promo_dates = (
  Window()
    .partitionBy('epos_product_id', 'epos_store_id')
    .orderBy(F.col('eow_date').cast('timestamp').cast('long'))
    .rangeBetween(-days(7 * utils.historical_promo_weeks), 0)
)
window_product_store = Window().partitionBy('epos_product_id', 'epos_store_id').orderBy(F.col('eow_date').asc()).rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

# Getting 0 or 1 based on whether the prediction date has a promo for a certain product-store combination
epos_data_with_requirements = (epos_data_with_requirements.withColumn("epos_price_per_product", F.col('epos_value')/F.col('epos_quantity'))
                                                           .withColumn("mean_unit_price", F.mean(F.col('epos_price_per_product')).over(window_promo_dates))
                                                           .withColumn("unit_price_difference", (F.col("epos_price_per_product") / F.col("mean_unit_price")) - 1)
                                                           .withColumn("is_promotion", F.when(F.col("unit_price_difference") < -(utils.promotion_percentage / 100.0), 1).otherwise(0))
                                                           .withColumn('promo_prediction_week', F.last('is_promotion').over(window_product_store))
                                                           .drop("epos_price_per_product", "mean_unit_price", "unit_price_difference")
                                ).persist()

In [0]:
window_products = Window().partitionBy('epos_product_id')

promo_product_df = (epos_data_with_requirements.filter(F.col('eow_date').isin(PREDICTION_DATES))
                        .groupBy('epos_product_id', 'is_promotion')
                        .agg(F.countDistinct('epos_store_id').alias('n_stores_group'))
                        .withColumn('n_rows', F.count('*').over(window_products))
                        .withColumn('total_stores', F.sum(F.col('n_stores_group')).over(window_products))
                        .withColumn("promo_group", F.when((F.col('n_stores_group') < 20) & (F.col('total_stores') - F.col('n_stores_group') < 20), 0).when(F.col('n_stores_group') < 20, 1 - F.col("is_promotion")).otherwise(F.col('is_promotion'))) # If both groups are < 20, set promo group at 0. If n_stores_group is <20 and for other >=20, set both groups to the biggest group. If group is >= 20, keep it's own promo_group
                        .withColumnRenamed("is_promotion", "promo_prediction_week")
                   )

epos_data_with_requirements = (epos_data_with_requirements.join(promo_product_df.select('epos_product_id', 'promo_prediction_week', 'promo_group'), on=['epos_product_id', 'promo_prediction_week'], how = 'left')
                                                         .withColumnRenamed("epos_product_id", "epos_product_id_original")
                                                         .withColumn('epos_product_id', F.concat(F.col("epos_product_id_original"), F.lit("_"), F.col("promo_group")))
                                                         .drop("epos_product_id_original", "promo_prediction_week", "promo_group")
                              ).persist()

In [0]:
if pre_work_feature_selection:
  # Calculate number of stores in which product is sold (n_stores_sold), rank these numbers based on percentiles (percent_rank), divide into 3 groups (store_group) 
  grouped_products = (
                    epos_data_with_requirements
                    .groupBy('epos_product_id')
                    .agg(F.countDistinct('epos_store_id').alias('n_stores_sold'), F.mean('epos_quantity').alias('avg_quantity_per_week'))
                    .withColumn('percent_rank_store', F.percent_rank().over(Window.partitionBy().orderBy('n_stores_sold')))
                    .withColumn('percent_rank_quantity', F.percent_rank().over(Window.partitionBy().orderBy('avg_quantity_per_week')))
                    .withColumn('store_group', F.when(F.col('percent_rank_store').between(0,0.333), 1)
                                                .when(F.col('percent_rank_store').between(0.333,0.667), 2)
                                                .when(F.col('percent_rank_store').between(0.667,1), 3))
                    .withColumn('quantity_group', F.when(F.col('percent_rank_quantity').between(0,0.333), 1)
                                                .when(F.col('percent_rank_quantity').between(0.333,0.667), 2)
                                                .when(F.col('percent_rank_quantity').between(0.667,1), 3))
                    .withColumn("is_promotion", F.substring(F.col("epos_product_id"), -1, 1))
                  ).persist()

  grouped_products.display()

In [0]:
if pre_work_feature_selection:
  product_list_promo = {}
  product_list_non_promo = {}

  for sg in range(1,4):
    for qg in range(1,4):
      try:
        promo_id = grouped_products.filter((F.col('store_group') == sg) & (F.col('quantity_group') == qg) & (F.col("is_promotion")=='1')).select('epos_product_id').rdd.takeSample(False, num=1, seed=1)[0][0]
        product_list_promo[promo_id] = str(sg) + "-" + str(qg)
      except IndexError:
        print('Promo: Store group: ' + str(sg) + ' and quantity group: ' + str(qg) + ' has no options')
      try:      
        non_promo_id = grouped_products.filter((F.col('store_group') == sg) & (F.col('quantity_group') == qg) & (F.col("is_promotion")=='0')).select('epos_product_id').rdd.takeSample(False, num=1, seed=1)[0][0]
        product_list_non_promo[non_promo_id] = str(sg) + "-" + str(qg)
      except IndexError:
        print('Non-promo: Store group: ' + str(sg) + ' and quantity group: ' + str(qg) + ' has no options')

In [0]:
import random
# Since promo group is smaller, first select 3 products
promo_ids = random.sample(product_list_promo.keys(), min(len(product_list_promo), 3))
promo_groups = [product_list_promo[product_id] for product_id in promo_ids]
# Select ids from non_promo that do not match the groups of promo ids
filtered_non_promo_list = { key:value for (key,value) in product_list_non_promo.items() if value not in promo_groups}
non_promo_ids = list(filtered_non_promo_list.keys())

print(promo_ids+non_promo_ids)